# Training Hub on Kubeflow — with unified callbacks

Based on [ODH distributed-workloads trainer notebooks](https://github.com/opendatahub-io/distributed-workloads/tree/main/tests/trainer/resources), plus `TrainingHubTrainer.callbacks` from kubeflow-sdk (RHOAIENG-79848).

**Callbacks:** pass a `TrainingHubCallback` **class** (not an instance). SDK serializes it into the training pod and merges into `sft` / `osft` / `lora_sft`.

Requires kubeflow-sdk with callback injection and training_hub with unified callbacks (77626/77627).


In [ ]:
%pip install datasets --quiet

import logging, os, warnings
logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")
warnings.filterwarnings("ignore")
os.environ.setdefault("HF_DATASETS_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("TQDM_DISABLE", "1")


In [ ]:
# Standard library imports
import logging
import os
import sys
import time
from io import StringIO


In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass


In [ ]:
import os
from kubernetes import client as k8s, config as k8s_config

# Resolve namespace from in-cluster service account or env var (no hardcode)
_ns_file = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"
NAMESPACE = (
    open(_ns_file).read().strip()
    if os.path.exists(_ns_file)
    else os.environ.get("POD_NAMESPACE", "")
)

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")

if api_server and token:
    configuration = k8s.Configuration()
    configuration.host = api_server
    configuration.verify_ssl = False
    configuration.api_key = {"authorization": f"Bearer {token}"}
    api_client = k8s.ApiClient(configuration)
else:
    k8s_config.load_incluster_config()
    api_client = k8s.ApiClient()
    print("Using in-cluster config")

# Resolve PVC name — auto-detect from this pod's volume mounts (no RBAC needed)
PVC_NAME = os.getenv("SHARED_PVC_NAME")
if not PVC_NAME:
    _pod_name = os.environ.get("HOSTNAME", "")
    if _pod_name:
        try:
            _v1 = k8s.CoreV1Api(api_client)
            _pod = _v1.read_namespaced_pod(name=_pod_name, namespace=NAMESPACE or "default")
            for vol in (_pod.spec.volumes or []):
                if vol.persistent_volume_claim:
                    PVC_NAME = vol.persistent_volume_claim.claim_name
                    print(f"Auto-detected PVC from pod volumes: {PVC_NAME}")
                    break
        except Exception as _e:
            print(f"Warning: could not read pod spec to auto-detect PVC: {_e}")
    if not PVC_NAME:
        raise RuntimeError(
            "SHARED_PVC_NAME env var is not set and could not auto-detect PVC.\n"
            "Set SHARED_PVC_NAME to the name of the PVC mounted to this notebook pod."
        )

PVC_MOUNT_PATH = "/opt/app-root/src"
print(f"Namespace: {NAMESPACE!r}, PVC: {PVC_NAME}")


In [ ]:
# Converting the format of the intial messages.
def convert_to_messages(example):
    """
    Convert a sql-create-context example to chat template format.
    
    The user provides the database schema and question.
    The assistant responds with the SQL query.
    """
    user_message = f"""Given the following database schema:

{example['context']}

Write a SQL query to answer this question: {example['question']}"""
    
    assistant_message = example['answer']
    
    return {
        "messages": [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]
    }


In [ ]:
import os
import gzip
import shutil
import time
import socket

try:
    import s3fs
    HAS_S3FS = True
except ImportError:
    HAS_S3FS = False

# --- Global networking safety net: cap all socket operations ---
socket.setdefaulttimeout(10)  # seconds
# Notebook's PVC mount path (per Notebook CR). Training pods will mount the same PVC at /opt/app-root/src
PVC_NOTEBOOK_PATH = "/opt/app-root/src/"
DATASET_ROOT_NOTEBOOK = PVC_NOTEBOOK_PATH
TXT_SQL_DIR = os.path.join(DATASET_ROOT_NOTEBOOK, "txt-sql-data", "train")
MODEL_DIR = os.path.join(DATASET_ROOT_NOTEBOOK, "Qwen", "Qwen2.5-1.5B-Instruct")
UNSLOTH_MODEL_DIR = os.path.join(DATASET_ROOT_NOTEBOOK, "unsloth", "qwen2.5-1.5b-instruct-unsloth-bnb-4bit")
os.makedirs(TXT_SQL_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(UNSLOTH_MODEL_DIR, exist_ok=True)

# Env config for S3/MinIO
s3_endpoint = os.getenv("AWS_DEFAULT_ENDPOINT", "")
s3_access_key = os.getenv("AWS_ACCESS_KEY_ID", "")
s3_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY", "")
s3_bucket = os.getenv("AWS_STORAGE_BUCKET", "")
s3_prefix = os.getenv("AWS_STORAGE_BUCKET_LORA_DIR", "")  

data_download_successful = False

if HAS_S3FS and s3_endpoint and s3_bucket:
    try:
        endpoint_url = (
            s3_endpoint
            if s3_endpoint.startswith("http")
            else f"https://{s3_endpoint}"
        )
        prefix = (s3_prefix or "").strip("/")

        print(
            f"[notebook] S3 configured: "
            f"endpoint={endpoint_url}, bucket={s3_bucket}, prefix={prefix or '<root>'}"
        )

        fs = s3fs.S3FileSystem(
            key=s3_access_key,
            secret=s3_secret_key,
            endpoint_url=endpoint_url,
            use_ssl=endpoint_url.startswith("https"),
            config_kwargs={"signature_version": "s3v4"},
            client_kwargs={"verify": False},
        )

        remote_path = f"{s3_bucket}/{prefix}" if prefix else s3_bucket
        pulled_any = False
        file_count = 0

        print(f"[notebook] Starting S3 download from prefix: {prefix}")
        for remote_file in fs.find(remote_path):
            file_count += 1

            if remote_file.endswith("/"):
                print(f"[notebook] Skipping directory marker: {remote_file}")
                continue

            rel = remote_file[len(remote_path):].lstrip("/")
            if not rel:
                continue
            print(f"[notebook] Processing rel={rel}")

            # Route to appropriate directory based on content type
            if rel.endswith(".jsonl") or "txt-sql" in rel.lower() or "sql-create" in rel.lower():
                dst = os.path.join(TXT_SQL_DIR, os.path.basename(rel))
                print(f"[notebook] Routing to dataset dir: {dst}")
            elif "unsloth" in rel.lower() or "bnb-4bit" in rel.lower():
                dst = os.path.join(UNSLOTH_MODEL_DIR, rel.split("qwen2.5-1.5b-instruct-unsloth-bnb-4bit/")[-1] if "qwen2.5-1.5b-instruct-unsloth-bnb-4bit" in rel else os.path.basename(rel))
                print(f"[notebook] Routing to unsloth model dir: {dst}")
            elif "qwen" in rel.lower() or (prefix and any(rel.endswith(ext) for ext in [".bin", ".json", ".model", ".safetensors", ".txt"])):
                dst = os.path.join(MODEL_DIR, rel.split("Qwen2.5-1.5B-Instruct/")[-1] if "Qwen2.5-1.5B-Instruct" in rel else os.path.basename(rel))
                print(f"[notebook] Routing to model dir: {dst}")
            else:
                if not prefix:
                    print(f"[notebook] Skipping unrelated file: {rel}")
                    continue
                dst = os.path.join(DATASET_ROOT_NOTEBOOK, rel)
                print(f"[notebook] Routing to default dir: {dst}")

            os.makedirs(os.path.dirname(dst), exist_ok=True)

            if not os.path.exists(dst):
                print(f"[notebook] Downloading s3://{remote_file} -> {dst}")
                t0 = time.time()
                fs.get(remote_file, dst)
                print(f"[notebook] DONE in {time.time() - t0:.2f}s")
                pulled_any = True
            else:
                print(f"[notebook] Skipping existing file {dst}")
                pulled_any = True

            # If the file is .gz, decompress and remove the .gz
            if dst.endswith(".gz") and os.path.exists(dst):
                out_path = os.path.splitext(dst)[0]
                if not os.path.exists(out_path):
                    print(f"[notebook] Decompressing {dst} -> {out_path}")
                    try:
                        with gzip.open(dst, "rb") as f_in, open(out_path, "wb") as f_out:
                            shutil.copyfileobj(f_in, f_out)
                    except Exception as e:
                        print(f"[notebook] Failed to decompress {dst}: {e}")
                    else:
                        try:
                            os.remove(dst)
                        except Exception:
                            pass

        if pulled_any:
            dataset_file = os.path.join(TXT_SQL_DIR, "train_All_100.jsonl")
            if os.path.exists(dataset_file):
                print(f"[notebook] ✓ S3 download successful. Processed {file_count} files")
                data_download_successful = True
            else:
                print(f"[notebook] S3 downloaded {file_count} files but dataset not found, will try HuggingFace fallback")
        else:
            print(f"[notebook] ✗ S3 download found no files to download")

    except Exception as e:
        print(f"[notebook] ✗ S3 fetch failed: {e}")
        import traceback
        traceback.print_exc()
        print("[notebook] Will attempt HuggingFace fallback...")
else:
    if not HAS_S3FS:
        print("[notebook] S3 not available: s3fs not installed")
    else:
        print("[notebook] S3 not configured (missing endpoint or bucket env vars)")

# Fallback to HuggingFace if S3 was not configured or failed (requires internet)
if not data_download_successful:
    print("[notebook] Attempting HuggingFace dataset download (requires internet)...")
    try:
        import json
        import random
        from datasets import load_dataset

        # Load the Table-GPT dataset
        print("[notebook] Loading Table-GPT dataset from HuggingFace...")
        # Load the dataset
        dataset = load_dataset("b-mc2/sql-create-context", split="train")

        TRAIN_SIZE = 100  # Adjust based on your time/compute budget

        # Shuffle and select a subset
        train_dataset = dataset.shuffle(seed=42).select(range(min(TRAIN_SIZE, len(dataset))))

        # Convert to messages format
        train_data = [convert_to_messages(example) for example in train_dataset]

        # Save the subset to a JSONL file
        output_file = os.path.join(TXT_SQL_DIR, "train_All_100.jsonl")
        with open(output_file, "w") as f:
            for example in train_data:
                f.write(json.dumps(example) + "\n")

        print(f"[notebook] ✓ HuggingFace download successful. Subset saved to {output_file}")
        data_download_successful = True

    except Exception as hf_error:
        print(f"[notebook] ✗ HuggingFace download failed: {hf_error}")
        import traceback
        traceback.print_exc()
        raise RuntimeError(
            "Failed to download dataset from both S3 and HuggingFace. "
            "In disconnected environments, ensure S3/MinIO is configured with the required data. "
            "In connected environments, check your internet connection and credentials."
        ) from hf_error

# Verify dataset file exists
dataset_file = os.path.join(TXT_SQL_DIR, "train_All_100.jsonl")
if os.path.exists(dataset_file):
    print(f"[notebook] ✓ Dataset ready: {dataset_file}")
else:
    raise RuntimeError(f"Dataset file not found: {dataset_file}")

# Verify model directory has files (model will be downloaded during training if not present)
if os.path.exists(MODEL_DIR) and os.listdir(MODEL_DIR):
    print(f"[notebook] ✓ Model files ready in: {MODEL_DIR}")
    print(f"[notebook] Model files: {os.listdir(MODEL_DIR)[:5]}...")  # Show first 5 files
else:
    print(f"[notebook] Note: Model directory is empty: {MODEL_DIR}")
    print("[notebook] Training will download model from HuggingFace during execution")


In [ ]:
# Model download - use S3 if available, otherwise HuggingFace
from huggingface_hub import snapshot_download

token = os.getenv("HUGGINGFACE_HUB_TOKEN")

# Base model (needed for tokenizer/config even when using Unsloth variant)
if os.path.exists(MODEL_DIR) and os.listdir(MODEL_DIR):
    print(f"✓ Using local base model from S3: {MODEL_DIR}")
else:
    print("[notebook] Base model not found in S3, downloading from HuggingFace...")
    snapshot_download(
        repo_id="Qwen/Qwen2.5-1.5B-Instruct",
        local_dir=MODEL_DIR,
        token=token,
        resume_download=True,
        local_dir_use_symlinks=False,
    )
    print(f"✓ Base model downloaded to: {MODEL_DIR}")

# Unsloth 4-bit model (used by LoRA with load_in_4bit=True)
if os.path.exists(UNSLOTH_MODEL_DIR) and os.listdir(UNSLOTH_MODEL_DIR):
    print(f"✓ Using local Unsloth model from S3: {UNSLOTH_MODEL_DIR}")
else:
    print("[notebook] Unsloth model not found in S3, downloading from HuggingFace...")
    snapshot_download(
        repo_id="unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit",
        local_dir=UNSLOTH_MODEL_DIR,
        token=token,
        resume_download=True,
        local_dir_use_symlinks=False,
    )
    print(f"✓ Unsloth model downloaded to: {UNSLOTH_MODEL_DIR}")


In [ ]:
# Use local Unsloth 4-bit model path (downloaded in previous cell if not from S3)
LOCAL_MODEL_PATH = "/opt/app-root/src/unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit"

# LoRA configuration
LORA_R = 16          # Rank - start small, increase if needed
LORA_ALPHA = 32      # Alpha - typically 2x rank
LORA_DROPOUT = 0.0   # Dropout - 0.0 is optimized for Unsloth

# Training configuration
NUM_EPOCHS = 2            # More epochs = better learning, longer training
LEARNING_RATE = 2e-4        # Standard LoRA learning rate
MAX_SEQ_LEN = 1024          # Maximum sequence length
MICRO_BATCH_SIZE = 16       # Batch size per GPU (reduce if OOM)
GRADIENT_ACCUMULATION = 4   # Effective batch = micro_batch * grad_accum

# QLoRA settings (set to True to enable 4-bit quantization)
USE_QLORA = True  # Set to True if you have limited GPU memory

params = {
        # Model and data path
        'model_path': LOCAL_MODEL_PATH,
        'data_path': "/opt/app-root/src/txt-sql-data/train/train_All_100.jsonl",
        'ckpt_output_dir': "/opt/app-root/src/checkpoints-logs-dir",
        'data_output_path': "/opt/app-root/src/lora-json/_data",
        # Important for LORA
        'lr_scheduler': "cosine",
        'warmup_steps': 0,
        'seed': 42,
        # LoRA configuration
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'lora_dropout': LORA_DROPOUT,

        # Training configuration
        'num_epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'micro_batch_size': MICRO_BATCH_SIZE,
        'max_seq_len': MAX_SEQ_LEN,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION,

        # Dataset format
        'dataset_type' : "chat_template",
        'field_messages' : "messages",
        # Quantization
        'load_in_4bit': USE_QLORA,
        # GPU configuration: configurable via NNODES env var, default 2
        'nnodes' : int(os.getenv("NNODES", "1")),
        # Logging
        'logging_steps': 10,
        'save_steps': 200,
        'save_total_limit': 3,

        # Model Checkpointing
        'save_final_checkpoint': True,
        'checkpoint_at_epoch': False,
}


In [ ]:
from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TrainingHubAlgorithms
from kubeflow.trainer.rhai import TrainingHubTrainer
from kubeflow.common.types import KubernetesBackendConfig

backend_cfg = KubernetesBackendConfig(client_configuration=api_client.configuration)
client = TrainerClient(backend_cfg)


In [ ]:
# Discover training runtimes — auto-select a training-hub runtime
training_runtime_name = os.getenv("TRAINING_RUNTIME")

runtimes = list(client.list_runtimes())
if not runtimes:
    raise RuntimeError("No training runtimes found in this cluster")

if training_runtime_name:
    th_runtime = next((r for r in runtimes if r.name == training_runtime_name), None)
    if th_runtime is None:
        available = [r.name for r in runtimes]
        raise RuntimeError(
            f"Runtime {training_runtime_name!r} not found.\n"
            f"Available: {available}\n"
            f"Set TRAINING_RUNTIME env var to one of the above."
        )
else:
    th_runtimes = sorted(
        [r for r in runtimes if "training-hub" in r.name and "cuda" in r.name],
        key=lambda r: len(r.name), reverse=True,
    ) or [r for r in runtimes if "training-hub" in r.name]
    if th_runtimes:
        th_runtime = th_runtimes[0]
    else:
        th_runtime = runtimes[0]
    if len(th_runtimes) > 1:
        print(f"Auto-selected runtime: {th_runtime.name} (from {len(th_runtimes)} training-hub runtimes)")
    else:
        print(f"Auto-selected runtime: {th_runtime.name}")
    print("Set TRAINING_RUNTIME env var to override.")

print(f"Using runtime: {th_runtime.name}")


In [ ]:
import shutil, sys
from pathlib import Path

# Callback .py must live on the PVC so inspect.getsource works in-pod.
# torch patches inspect.getfile — importlib.util loading breaks getsource,
# so we use a normal sys.path import instead.
PVC_CALLBACK = Path("/opt/app-root/src/kubeflow_metrics_logger.py")
CANDIDATES = [
    Path("/opt/app-root/src/training_hub/examples/callbacks/kubeflow_metrics_logger.py"),
    Path("/opt/app-root/src/training_hub_checkout/examples/callbacks/kubeflow_metrics_logger.py"),
    Path("kubeflow_metrics_logger.py"),
]

src = next((p for p in CANDIDATES if p.is_file()), None)
if src is None and not PVC_CALLBACK.is_file():
    raise FileNotFoundError(
        "Copy examples/callbacks/kubeflow_metrics_logger.py to the PVC or clone training_hub. "
        f"Tried: {[str(p) for p in CANDIDATES]}"
    )
if src is not None and src.resolve() != PVC_CALLBACK.resolve():
    shutil.copy(src, PVC_CALLBACK)

if str(PVC_CALLBACK.parent) not in sys.path:
    sys.path.insert(0, str(PVC_CALLBACK.parent))
from kubeflow_metrics_logger import KubeflowMetricsLogger
print(f"Callback class ready: {KubeflowMetricsLogger.__name__}")


In [ ]:
from kubeflow.trainer.options import (
    RuntimePatch,
    TrainingRuntimeSpecPatch,
    JobSetTemplatePatch,
    JobSetSpecPatch,
    ReplicatedJobPatch,
    JobTemplatePatch,
    JobSpecPatch,
    PodTemplatePatch,
    PodSpecPatch,
    ContainerPatch,
)

cache_root = "/opt/app-root/src/.cache/huggingface"
triton_cache = "/tmp/.triton"

job_name = client.train(
    trainer=TrainingHubTrainer(
        algorithm=TrainingHubAlgorithms.LORA_SFT,
        func_args=params,
        env={
            "HF_HOME": cache_root,
            "TRITON_CACHE_DIR": triton_cache,
            "XDG_CACHE_HOME": "/opt/app-root/src/.cache",
            "NCCL_DEBUG": "ERROR",
            "PYTHONWARNINGS": "ignore",
            "LOG_LEVEL": "ERROR",
            "ACCELERATE_LOG_LEVEL": "error",
            "HF_DATASETS_DISABLE_PROGRESS_BARS": "1",
            "HF_HUB_DISABLE_PROGRESS_BARS": "1",
            "TRANSFORMERS_VERBOSITY": "error",
            "TQDM_DISABLE": "1",
            "BITSANDBYTES_NOWELCOME": "1",
            "TOKENIZERS_PARALLELISM": "false",
        },
        callbacks=[KubeflowMetricsLogger],
        resources_per_node={"cpu": 4, "memory": "32Gi", "nvidia.com/gpu": 1},
    ),
    options=[
        RuntimePatch(
            training_runtime_spec=TrainingRuntimeSpecPatch(
                template=JobSetTemplatePatch(
                    spec=JobSetSpecPatch(
                        replicated_jobs=[
                            ReplicatedJobPatch(
                                name="node",
                                template=JobTemplatePatch(
                                    spec=JobSpecPatch(
                                        template=PodTemplatePatch(
                                            spec=PodSpecPatch(
                                                volumes=[
                                                    {"name": "work", "persistentVolumeClaim": {"claimName": PVC_NAME}},
                                                ],
                                                containers=[
                                                    ContainerPatch(
                                                        name="node",
                                                        volume_mounts=[
                                                            {"name": "work", "mountPath": "/opt/app-root/src", "readOnly": False},
                                                        ],
                                                    )
                                                ],
                                            )
                                        )
                                    )
                                )
                            )
                        ]
                    )
                ),
            )
        )
    ],
    runtime=th_runtime,
)
print(f"Training job created: {job_name}")


In [ ]:
# Wait for the running status, then wait for completion or failure
# Using reasonable timeout for LORA training
client.wait_for_job_status(name=job_name, status={"Running"}, timeout=600)
client.wait_for_job_status(name=job_name, status={"Complete", "Failed"}, timeout=1800)  # 30 minutes for training

# Get job details and logs
job = client.get_job(name=job_name)
pod_logs = client.get_job_logs(name=job_name, follow=False)

logs = []
for log_line in pod_logs:
    logs.extend(str(log_line).splitlines())
log_text = "\n".join(logs)

print(f"Training job final status: {job.status}")

# Check 1: Job status must not be "Failed"  
if job.status == "Failed":
    print(f"ERROR: Training job '{job_name}' has Failed status")
    print("Last 30 lines of logs:")
    for line in logs[-30:]:
        print(line)
    raise RuntimeError(f"Training job '{job_name}' failed")

# Check 2: Look for the training completion message in logs
# This is critical because the training script may catch exceptions and exit 0
if "[PY] LORA_SFT training complete. Result=" not in log_text:
    print(f"ERROR: Training completion message not found in logs")
    print("Last 50 lines of logs:")
    for line in logs[-50:]:
        print(line)
    raise RuntimeError(f"Training did not complete successfully - missing completion message")

# Check 3: unified callback hooks appeared in logs
if "[TH-CB]" not in log_text:
    print("WARNING: No [TH-CB] callback log lines found — verify kubeflow-sdk callback injection")
else:
    cb_lines = [line.strip() for line in logs if "[TH-CB]" in line]
    print(f"✓ Callbacks ({len(cb_lines)}): {' | '.join(cb_lines)}")

print(f"✓ Training job '{job_name}' completed successfully")


In [ ]:
for c in client.get_job(name=job_name).steps:
    print(f"Step: {c.name}, Status: {c.status}, Devices: {c.device} x {c.device_count}\n")


In [ ]:
# Full pod logs are verbose — uncomment only when debugging.
# for log_line in client.get_job_logs(name=job_name, follow=False):
#     print(log_line)


In [ ]:
client.delete_job(job_name)
